# Artigo 0005 - Prompt Engineering Avancado

Notebook com benchmark real usando modelos gratuitos do Hugging Face.

## Modelos usados
- Geração: `Qwen/Qwen2.5-0.5B-Instruct` e `google/flan-t5-small`
- Embeddings para score semântico: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

Aqui comparamos quatro estratégias em dois modelos gratuitos:
1. prompt genérico
2. prompt estruturado
3. prompt few-shot
4. prompt com checklist

In [ ]:
from pathlib import Path
import sys

# Garante que o notebook consiga importar o pacote local tanto
# quando for executado dentro da pasta do artigo quanto via raiz do repositório.
ARTICLE_ROOT = Path.cwd()
if not (ARTICLE_ROOT / "src").exists():
    ARTICLE_ROOT = ARTICLE_ROOT.parent
sys.path.insert(0, str(ARTICLE_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.prompt_benchmark import (
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_GENERATION_MODELS,
    run_benchmark,
)

## Dataset do laboratório

O benchmark só faz sentido porque cada caso já traz:

- o texto de entrada;
- um resumo ideal esperado;
- a prioridade esperada;
- keywords obrigatórias que não podem sumir da resposta.

In [ ]:
dataset_preview = pd.read_json(ARTICLE_ROOT / "data" / "prompt_dataset.json")

# Cada linha funciona como um caso de benchmark. Esse preview ajuda a validar
# se o experimento cobre suporte, financeiro e contexto comercial.
dataset_preview[["id", "expected_priority", "required_keywords"]]

In [ ]:
# Executa o laboratório completo em um subconjunto curto para iteração rápida.
# Para usar o dataset inteiro, basta remover o parâmetro limit.
resultados, resumo = run_benchmark(limit=2)
print("Modelos de geracao:", ", ".join(DEFAULT_GENERATION_MODELS))
print("Modelo de embedding:", DEFAULT_EMBEDDING_MODEL)
resumo


## O que saiu do laboratório

A tabela abaixo mostra a granularidade completa do experimento. Em vez de olhar apenas para uma média final, conseguimos enxergar:

- qual modelo foi executado;
- qual estratégia foi aplicada;
- como cada dimensão do score reagiu;
- em quais casos a resposta ainda ficou incompleta.

In [ ]:
resultados[[
    "modelo_geracao",
    "caso",
    "estrategia",
    "score_estrutura",
    "score_keywords",
    "score_prioridade",
    "score_semantico",
    "score_total",
]]

In [ ]:
pivot_total = resumo.pivot(index="estrategia", columns="modelo_geracao", values="score_total")
ax = pivot_total.plot(
    kind="bar",
    figsize=(10, 4),
    ylim=(0, 1.05),
    title="Score total por estrategia e por modelo",
)
ax.set_ylabel("score_total")
plt.xticks(rotation=0)
plt.show()

In [ ]:
for model_name in DEFAULT_GENERATION_MODELS:
    bloco = resumo[resumo["modelo_geracao"] == model_name]
    base_score = float(bloco.loc[bloco["estrategia"] == "base", "score_total"].iloc[0])
    melhor = bloco.sort_values("score_total", ascending=False).iloc[0]
    ganho = melhor["score_total"] - base_score
    ganho_pct = (ganho / base_score) * 100 if base_score else 0
    print(f"{model_name}: melhor estrategia = {melhor['estrategia']} | ganho = +{ganho:.3f} pontos ({ganho_pct:.1f}%)")

In [ ]:
amostra = resultados[resultados["caso"] == resultados["caso"].iloc[0]][["estrategia", "saida", "score_total"]]
amostra

## Artefatos auxiliares do benchmark

O script principal também grava arquivos auxiliares para análise fora do notebook:

- `benchmark_deltas.csv` com o delta de cada estratégia contra a baseline dentro do mesmo modelo;
- `benchmark_case_breakdown.csv` com score por caso e campos ausentes;
- `benchmark_relatorio.md` com um resumo executivo.

In [ ]:
delta_path = ARTICLE_ROOT / "data" / "benchmark_deltas.csv"
breakdown_path = ARTICLE_ROOT / "data" / "benchmark_case_breakdown.csv"

delta = pd.read_csv(delta_path)
breakdown = pd.read_csv(breakdown_path)

# Delta por modelo e estratégia. Aqui fica claro quem melhorou de fato
# em relação à baseline e quem só pareceu melhor em um caso isolado.
delta[["modelo_geracao", "estrategia", "score_total", "delta_vs_base", "delta_vs_base_pct"]]

In [ ]:
# Breakdown por caso. Este recorte é útil para descobrir em que cenário
# o modelo falhou em estrutura, perdeu keyword ou deixou campo obrigatório vazio.
breakdown.sort_values(["modelo_geracao", "score_total"], ascending=[True, False])[ [
    "modelo_geracao",
    "caso",
    "estrategia",
    "score_total",
    "missing_fields",
]]